RAG Pipeline =data ingestion to VectorDB pipeline


In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader, TextLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from  pathlib import Path
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI          # ← replaces ChatGroq & openrouter

load_dotenv()

True

In [2]:
# Initialize OpenRouter LLM via OpenAI-compatible interface
OpenRouter_API_KEY = os.getenv("OpenRouter_API_KEY")

# llm = ChatOpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=OpenRouter_API_KEY,
#     model="nvidia/nemotron-3-super-120b-a12b:free",
#     temperature=0.1,
#     max_tokens=1024,
# )
# print(f"LLM initialized: {llm.model_name}")

In [3]:
def process_all_pdf(pdf_directory):
    all_doc=[]
    pdf_dir=Path(pdf_directory)
    pdf_files=list(pdf_dir.glob("*.pdf"))
    for pdf_file in pdf_files:
        print(f"processing{pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))
            documents=loader.load()
            # add source info to metadata
            for doc in documents:
                doc.metadata["Source"]=pdf_file.name
                doc.metadata["File_Type"]="pdf"
            all_doc.extend(documents)
            print(f"loaded {len(documents)}")
        except Exception as e:
            print(f"error loading {pdf_file.name}:{e}")
        print(f"\n total documents loaded:{len(all_doc)}")
    return all_doc
all_pdf_docs=process_all_pdf("../data/text_files")

processingregression_detailed.pdf
loaded 11

 total documents loaded:11
processingunsupervised_detailed.pdf
loaded 12

 total documents loaded:23
processingclassification_detailed.pdf
loaded 12

 total documents loaded:35
processingdeep_learning_master.pdf
loaded 28

 total documents loaded:63


In [4]:
all_pdf_docs


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-01T11:18:59+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-01T11:18:59+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/text_files/regression_detailed.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'Source': 'regression_detailed.pdf', 'File_Type': 'pdf'}, page_content='Complete Guide to Regression Models\n1. Regression is a fundamental area in machine learning. This section explains theory, intuition,\nalgorithms, math foundations, preprocessing, feature engineering, model selection, evaluation\nmetrics, real-world applications, advantages, limitations, optimization strategies, and practical\nexamples. 2. Regression is a fundamental area in machine learning. This section explains theory,\nintuition, algorithms, math foundations, preprocessing, feature engineering, model selectio

In [5]:
# text splitting
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=chunk_size,
                                                 chunk_overlap=chunk_overlap,
                                                 length_function=len,
                                                 separators=["\n\n","\n","."," ",""]
                                                 )
    split_docs=text_splitter.split_documents(documents)
    print(f"split into {len(split_docs)} chunks")

# EXAMPLE
    if split_docs:
        print(f"first chunk:\n {split_docs[0].page_content}\n")
        print(f"metadata:\n {split_docs[0].metadata}")
    return split_docs

In [6]:
chunks=split_documents(all_pdf_docs)

split into 434 chunks
first chunk:
 Complete Guide to Regression Models
1. Regression is a fundamental area in machine learning. This section explains theory, intuition,
algorithms, math foundations, preprocessing, feature engineering, model selection, evaluation
metrics, real-world applications, advantages, limitations, optimization strategies, and practical
examples. 2. Regression is a fundamental area in machine learning. This section explains theory,
intuition, algorithms, math foundations, preprocessing, feature engineering, model selection,
evaluation metrics, real-world applications, advantages, limitations, optimization strategies, and
practical examples. 3. Regression is a fundamental area in machine learning. This section explains
theory, intuition, algorithms, math foundations, preprocessing, feature engineering, model selection,
evaluation metrics, real-world applications, advantages, limitations, optimization strategies, and

metadata:
 {'producer': 'ReportLab PDF Library 

In [7]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-01T11:18:59+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-01T11:18:59+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/text_files/regression_detailed.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'Source': 'regression_detailed.pdf', 'File_Type': 'pdf'}, page_content='Complete Guide to Regression Models\n1. Regression is a fundamental area in machine learning. This section explains theory, intuition,\nalgorithms, math foundations, preprocessing, feature engineering, model selection, evaluation\nmetrics, real-world applications, advantages, limitations, optimization strategies, and practical\nexamples. 2. Regression is a fundamental area in machine learning. This section explains theory,\nintuition, algorithms, math foundations, preprocessing, feature engineering, model selectio


embeddigs And vector Db

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()

# initialize the embbeding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 384


/tmp/ipykernel_6621/4248987900.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [10]:
import os
import uuid
from typing import List, Any
import numpy as np
import chromadb


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./data/vector_store"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vectorstore=VectorStore()
vectorstore  



Vector store initialized. Collection: pdf_documents
Existing documents in collection: 6944


In [11]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-01T11:18:59+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-01T11:18:59+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/text_files/regression_detailed.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'Source': 'regression_detailed.pdf', 'File_Type': 'pdf'}, page_content='Complete Guide to Regression Models\n1. Regression is a fundamental area in machine learning. This section explains theory, intuition,\nalgorithms, math foundations, preprocessing, feature engineering, model selection, evaluation\nmetrics, real-world applications, advantages, limitations, optimization strategies, and practical\nexamples. 2. Regression is a fundamental area in machine learning. This section explains theory,\nintuition, algorithms, math foundations, preprocessing, feature engineering, model selectio

In [12]:
# ##convert  the text chunks to embeddings
text=[doc.page_content for doc in chunks]
# generate the embeddings
embeddings=embedding_manager.generate_embeddings(text)

# ?store in vectordb
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 434 texts...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Generated embeddings with shape: (434, 384)
Adding 434 documents to vector store...
Successfully added 434 documents to vector store
Total documents in collection: 7378


RAG Retriever Pipeline

In [ ]:
from typing import List, Dict, Any


class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
rag_retriever = RAGRetriever(vector_store=vectorstore, embedding_manager=embedding_manager)

In [14]:
rag_retriever.retrieve("What is a regression analysis?")

Retrieving documents for query: 'What is a regression analysis?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_8fbc8d4e_48',
  'content': 'Regression is a fundamental area in machine learning. This section explains theory, intuition,\nalgorithms, math foundations, preprocessing, feature engineering, model selection, evaluation\nmetrics, real-world applications, advantages, limitations, optimization strategies, and practical\nexamples. 119. Regression is a fundamental area in machine learning. This section explains\ntheory, intuition, algorithms, math foundations, preprocessing, feature engineering, model selection,\nevaluation metrics, real-world applications, advantages, limitations, optimization strategies, and\npractical examples. 120. Regression is a fundamental area in machine learning. This section\nexplains theory, intuition, algorithms, math foundations, preprocessing, feature engineering, model\nselection, evaluation metrics, real-world applications, advantages, limitations, optimization\nstrategies, and practical examples. 121. Regression is a fundamental area in machine 

In [15]:

from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage


In [16]:
class openrouterLLM:
    def __init__(self, model_name: str = "nvidia/nemotron-3-super-120b-a12b:free", api_key: str =None):
        """
        Initialize openrouter LLM
        
        Args:
            model_name: open router model name (nvidia/nemotron-3-super-120b-a12b:free etc.)
            api_key: OpenRouter API key (or set OpenRouter_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("OpenRouter_API_KEY")
        
        if not self.api_key:
            raise ValueError("OpenRouter API key is required. Set OpenRouter_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatOpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=self.api_key,
            model=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized OpenRouter LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [17]:


# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    openrouterLLM = openrouterLLM(api_key=os.getenv("OpenRouter_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None



Initialized OpenRouter LLM with model: nvidia/nemotron-3-super-120b-a12b:free
Groq LLM initialized successfully!


In [18]:
### get the context from the retriever and pass it to the LLM

rag_retriever.retrieve("Machine learnig models")

Retrieving documents for query: 'Machine learnig models'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_3780aec2_121',
  'content': 'model selection, evaluation metrics, real-world applications, advantages, limitations, optimization\nstrategies, and practical examples. 103. Unsupervised Learning is a fundamental area in machine\nlearning. This section explains theory, intuition, algorithms, math foundations, preprocessing,\nfeature engineering, model selection, evaluation metrics, real-world applications, advantages,\nlimitations, optimization strategies, and practical examples. 104. Unsupervised Learning is a\nfundamental area in machine learning. This section explains theory, intuition, algorithms, math\nfoundations, preprocessing, feature engineering, model selection, evaluation metrics, real-world\napplications, advantages, limitations, optimization strategies, and practical examples. 105.',
  'metadata': {'author': '(anonymous)',
   'page': 5,
   'title': '(anonymous)',
   'File_Type': 'pdf',
   'Source': 'unsupervised_detailed.pdf',
   'producer': 'ReportLab PDF Librar

In [19]:
### Simple RAG pipeline with OpenRouter LLM


### Initialize the OpenRouter LLM (set your OpenRouter_API_KEY in environment)
OpenRouter_API_KEY = os.getenv("OpenRouter_API_KEY")

llm=ChatOpenAI(base_url="https://openrouter.ai/api/v1",api_key=OpenRouter_API_KEY,model="nvidia/nemotron-3-super-120b-a12b:free",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using OpenRouter LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [20]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [21]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is Machine learning", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])


Retrieving documents for query: 'what is Machine learning'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
fundamental area in machine learning. This section explains theory, intuition, algorithms, math
foundations, preprocessing, feature engineering, model selection, evaluation metrics, real-world
applications, advantages, limitations, optimization strategies, and practical examples.

fundamental area in machine learning. This section explains theory, intuition, algorithms, math
foundations, preprocessing, feature engineering, model selection, evaluation metrics, real-world
applications, advantages, limitations, optimization strategies, and practical examples.

fundamental area in machine learning. This section explains theory, intuition, algorithms, math
foundations, preprocessing, feature engineering, model selection, evaluation metrics, real-world
applications, advantages, limitations, optimization strategies, and practi